This notebook is used to load and test the HRS RESPONDENT table.  It extracts distinct HHIDPN values from the RAND longitudinal data and populate the table.

**Purpose:** Load the HRS RESONDENT reference table.

**Source Table:** `dev_catalog.brz_raw_hrs.randhrs1992_2022v1`  
**Target Table:** `dev_catalog.slv_cdm_hrs.resondent`
**Load Script:** `../../sql/dml/load_hrs_respondent_data.sql`
**Validation Script:** `../../sql/validataion/verify_hrs_respondent_data.sql`

**Process:**
1. Clear/truncate the HRS RESONDENT table .
2. Extract distinct HHIDPN values from SOURCE data and load the TARGET table.
3. Validate the table data.
4. Display summary stats.

In [ ]:
# -----------------------------------------------------------------------------
# Initialize Notebook Configuration
# -----------------------------------------------------------------------------

dbutils.widgets.dropdown(
    "truncate_table",
    "true",
    ["true", "false"]
)

TRUNCATE_TABLE = dbutils.widgets.get("truncate_table").lower() == "true"

TARGET_TABLE = "dev_catalog.slv_cdm_hrs.hrs_respondent"

LOAD_SQL = "../../sql/dml/load_hrs_respondent_data.sql"

VALIDATION_SQL = "../../sql/validation/verify_hrs_respondent_data.sql"

SOURCE_TABLE = "dev_catalog.brz_raw_hrs.randhrs1992_2022v1"

In [ ]:
# Step 1:
# Clear existing respondent data if needed (use with caution)
# Uncomment the line below to truncate the table before loading

if TRUNCATE_TABLE:
    print("======================================================")
    print("Step 1 - TRUNCATE")
    print("======================================================")

    spark.sql(f"TRUNCATE TABLE {TARGET_TABLE}")
    print("✓ Completed")

else:
    print("Skipping table truncation.")

In [ ]:
# Step 2
# Load distinct HRS data to the TARGET_TABLE
import sys

# importlib to eliminate cache issues.
import importlib
import src.common.sql_utils as sql_utils

importlib.reload(sql_utils)

from src.common.sql_utils import execute_sql_file

print("======================================================")
print("Step 2 - LOAD DATA")
print("======================================================")
try: 
    execute_sql_file(
        spark, 
        LOAD_SQL
    )
    print("✓ Completed")
except Exception as e:
    print(f"❌ Load failed: {e}")
    raise


In [ ]:
# # Step 3: Verify the TARGET_TABLE

print("======================================================")
print("Step 3 - Validation")
print("======================================================")
execute_sql_file(
    spark,
    VALIDATION_SQL,
    display_results=True
)

print("✓ Completed")

In [ ]:
# Step 3: Verify the TARGET_TABLE

print("======================================================")
print("Step 3 - Validation")
print("======================================================")

# Read and execute the SQL file (handles multiple statements)
with open(VALIDATION_SQL, "r") as f:
    sql_content = f.read()
    
    # Split by semicolons and execute each statement
    statements = [s.strip() for s in sql_content.split(';') if s.strip()]
    
    for i, stmt in enumerate(statements, 1):
        print(f"\nExecuting statement {i}...")
        result_df = spark.sql(stmt)
        display(result_df)

print("\n✓ Completed")

In [ ]:
# Step 4 - Display summary statistics
DISTINCT_COL = "HHIDPN"

TARGET_TABLE = "respondent"

source_count = spark.sql("""
    SELECT COUNT(DISTINCT {DISTINCT_COL}) as distinct_{DISTINCT_COL}
    FROM dev_catalog.brz_raw_hrs.randhrs1992_2022v1
    WHERE HHIDPN IS NOT NULL
""").collect()[0][0]

target_count = spark.sql("""
    SELECT COUNT(*) as {TARGET_TABLE}_count
    FROM dev_catalog.slv_cdm_hrs.hrs_{TARGET_TABLE}
""").collect()[0][0]

print("=" * 60)
print("HRS {DISTINCT_COL} DATA LOAD SUMMARY")
print("=" * 60)
print(f"Distinct {DISTINCT_COL} values in source: {source_count}")
print(f"Total records in HRS {TARGET_TABLE} table:      {target_count}")
print("=" * 60)

if source_count == target_count:
    print("✓ SUCCESS: All distinct HRS {DISTINCT_COL} loaded")
else:
    print(f"⚠ WARNING: Mismatch detected. Please review.")

: 